# Step 3 — Offline baselines (ZOH / Linear / PID)

Fills the **missing** offline comparison table: upsample sparse VLA joint targets to 100 Hz **without** an ESN, then score against demo joints.

| Method | Meaning |
|--------|---------|
| `zoh` | Hold last VLA target (staircase) |
| `linear` | Straight-line blend between VLA knots |
| `pid` | Simple joint PID toward held VLA targets |

Does **not** load UnifoLM. Uses wipe-table episode data (same as Step 2).

Live closed-loop with real UnifoLM → `step3_dual_thread_mujoco.ipynb` (`BRIDGE=zoh|linear|esn`).


In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step3_control_baselines.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step1_baselines'}")


In [ ]:
# ── Configuration ───────────────────────────────────────────
EPISODE = 0
METHODS = ["zoh", "linear", "pid"]   # or subset, e.g. ["zoh", "linear"]

print(f"episode={EPISODE} | methods={METHODS}")


In [ ]:
import json
from dataclasses import asdict

import pandas as pd
from IPython.display import display

from src.paths import results_path
from src.step3_control_baselines import run_baseline, write_comparison_table

out_dir = results_path("step1_baselines")
eval_dir = results_path("step3_evaluation")
out_dir.mkdir(parents=True, exist_ok=True)
eval_dir.mkdir(parents=True, exist_ok=True)

results = []
for method in METHODS:
    r = run_baseline(method, EPISODE)
    results.append(r)
    path = out_dir / f"baseline_{method}_ep{EPISODE}.json"
    path.write_text(json.dumps(asdict(r), indent=2))
    print(f"{method:7s}  RMSE={r.rmse:.6f} rad  jerk={r.jerk:.3e}  → {path.name}")

csv_path = write_comparison_table(results, out_dir)
write_comparison_table(results, eval_dir)
print(f"Comparison CSV: {csv_path}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import asdict
from IPython.display import display

df = pd.DataFrame([asdict(r) for r in results])
display(df[["method", "episode", "rmse", "jerk", "jerk_rms"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(df["method"], df["rmse"], color=["#F44336", "#FF9800", "#9E9E9E"][: len(df)])
axes[0].set_ylabel("Joint RMSE (rad)")
axes[0].set_title(f"Offline baselines vs demo (ep {EPISODE})")
axes[1].bar(df["method"], df["jerk"], color=["#F44336", "#FF9800", "#9E9E9E"][: len(df)])
axes[1].set_ylabel("Jerk metric")
axes[1].set_title("Smoothness (lower often better)")
plt.tight_layout()
fig_path = out_dir / f"baseline_comparison_ep{EPISODE}.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Figure saved: {fig_path}")
print("Next: live UnifoLM comparison in step3_dual_thread_mujoco.ipynb (MOCK=False).")
